In [ ]:
# Install haystack with Chroma and OpenAI integration
# %pip install chroma-haystack haystack-ai trafilatura
# %pip install 'farm-haystack[all]'

In [1]:
%pip install haystack-ai 

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install chroma-haystack


Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [4]:
from dotenv import load_dotenv
import os

# Load .env file
load_dotenv()

# Access the OpenAI API Key
openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("Please set your OPENAI_API_KEY in the .env file.")

# (Optional) Confirm it's loaded (don't do this in production for security)
print(f"Loaded OpenAI Key: {openai_api_key[:5]}... (truncated for safety)")

Loaded OpenAI Key: sk-pr... (truncated for safety)


In [5]:
import os
import httpx

def ensure_tenant_and_database():
    chroma_host = "http://localhost:8800"
    chroma_token = os.getenv("CHROMA_SERVER_AUTHN_CREDENTIALS")

    if not chroma_token:
        raise ValueError("CHROMA_SERVER_AUTHN_CREDENTIALS is missing from environment variables!")

    headers = {"Authorization": f"Bearer {chroma_token}"}

    try:
        # Check if tenant exists
        resp = httpx.get(f"{chroma_host}/api/v2/tenants/default_tenant", headers=headers)
        if resp.status_code == 404:
            print("🔧 Creating tenant: default_tenant")
            create_resp = httpx.post(
                f"{chroma_host}/api/v2/tenants",
                headers=headers,
                json={"name": "default_tenant"}
            )
            create_resp.raise_for_status()

        # Check if database exists
        resp = httpx.get(f"{chroma_host}/api/v2/tenants/default_tenant/databases/default", headers=headers)
        if resp.status_code == 404:
            print("🔧 Creating database: default")
            create_resp = httpx.post(
                f"{chroma_host}/api/v2/tenants/default_tenant/databases",
                headers=headers,
                json={"name": "default"}
            )
            create_resp.raise_for_status()

        print("✅ Tenant and database are ready.")

    except httpx.HTTPStatusError as http_err:
        print(f"❌ HTTP error when ensuring tenant/database: {http_err}")
        print(f"Response: {http_err.response.text}")
        raise

    except httpx.RequestError as req_err:
        print(f"❌ Network error when talking to Chroma: {req_err}")
        raise

    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        raise


# Call this once before creating the document store
ensure_tenant_and_database()


🔧 Creating database: default
✅ Tenant and database are ready.


In [6]:
import os
from haystack_integrations.document_stores.chroma import ChromaDocumentStore
import chromadb
import chromadb.api

class AuthenticatedChromaDocumentStore(ChromaDocumentStore):
    def _ensure_initialized(self):
        if not self._initialized:
            chroma_token = os.getenv('CHROMA_SERVER_AUTHN_CREDENTIALS')
            if not chroma_token:
                raise ValueError("CHROMA_SERVER_AUTHN_CREDENTIALS is missing in environment!")

            auth_headers = {
                "Authorization": f"Bearer {chroma_token}"
            }

            if self._host and self._port is not None:
                chromadb.api.client.SharedSystemClient.clear_system_cache()
                
                self.client = chromadb.HttpClient(
                    host=self._host,
                    port=self._port,
                    headers=auth_headers,  # Set headers
                    tenant="default_tenant",   # Set the tenant
                    database="default"         # Set the database
                )
            elif self._persist_path:
                self.client = chromadb.PersistentClient(path=self._persist_path)
            else:
                self.client = chromadb.Client()

            if self._collection_name in [col.name for col in self.client.list_collections()]:
                self._collection = self.client.get_collection(self._collection_name, embedding_function=self._embedding_func)
            else:
                self._collection = self.client.create_collection(name=self._collection_name, embedding_function=self._embedding_func)

            self._initialized = True


In [11]:
from haystack import Document

# Important: Use haystack_integrations for Haystack 2.x
from haystack_integrations.document_stores.chroma import ChromaDocumentStore
from haystack.components.embedders import OpenAIDocumentEmbedder
from haystack.utils import Secret


# 1️⃣ Read data from file.txt
with open("data/test.txt", "r", encoding="utf-8") as file:
    file_content = file.read()
# print (file_content)

# 2️⃣ Initialize OpenAI Embedder
document_embedder = OpenAIDocumentEmbedder(
    api_key=Secret.from_token(openai_api_key),
    model="text-embedding-ada-002"  # Change if you want to use a different model
)

# 3️⃣ Create Document and Embed it
document = Document(content=file_content)
embedded_docs = document_embedder.run([document])
# Extract documents from result dict
documents = embedded_docs["documents"]
document = documents[0]
# print(document) 

# 4️⃣ Create Chroma Document Store (remote or local)
document_store = AuthenticatedChromaDocumentStore(
    collection_name="documents",
    host="localhost",  # Adjust if you use a remote Chroma server
    port=8800          # Default port for Chroma server
)

# 5️⃣ Store Document into Chroma
document_store.write_documents([document])
# collection = document_store.get_or_create_collection(name="documents")
# collection.add(
#     documents=[document],
#     ids=["doc1"]
# )

print("Document embedded and stored successfully!")

Calculating embeddings: 100%|██████████| 1/1 [00:00<00:00,  1.33it/s]


NotImplementedError: In Chroma v0.6.0, list_collections only returns collection names. Use Client.get_collection(documents) to access name. See https://docs.trychroma.com/deployment/migration for more information.

In [ ]:
from haystack.components.embedders import OpenAITextEmbedder

query_embedder = OpenAITextEmbedder(
    api_key=Secret.from_token(openai_api_key),
    model="text-embedding-ada-002"
)

# 6️⃣ Query Example
query = "What is this file about?"
query_embedding = query_embedder.run(query)

# Search for the most relevant document
results = document_store.search_embeddings([query_embedding], top_k=3)


# # 7️⃣ Display Results
# print("\n🔍 Search Results:")
# for doc in results[0]:
#     print(f"- Content: {doc.content[:200]}...")  # print first 200 chars

NotImplementedError: In Chroma v0.6.0, list_collections only returns collection names. Use Client.get_collection(documents) to access name. See https://docs.trychroma.com/deployment/migration for more information.